In [ ]:
# ======================================
# Logistic Regression - Pancreatic Cancer Prediction
# (With Overfitting Check)
# ======================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# --------------------------------------
# LOAD DATASET
# --------------------------------------
df = pd.read_csv('/content/pancreatic_600_risk123.csv')  # change path for VS Code

# Drop irrelevant columns
df.drop(columns=['sample_id', 'patient_cc', 'sample_o'], inplace=True)

# --------------------------------------
# HANDLE MISSING VALUES
# --------------------------------------
num_cols = df.select_dtypes(include=['int64','float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Encode sex
df['sex'] = df['sex'].map({'M': 0, 'F': 1})

# Convert diagnosis to binary
df['diagnosis'] = df['diagnosis'].apply(lambda x: 1 if x == 3 else 0)

# --------------------------------------
# SPLIT DATA
# --------------------------------------
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# --------------------------------------
# FEATURE SCALING
# --------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------
# TRAIN LOGISTIC REGRESSION (WITH REGULARIZATION)
# --------------------------------------
lr = LogisticRegression(
    max_iter=1000,
    C=0.5,              # regularization strength (lower = stronger)
    penalty='l2',
    solver='liblinear'
)

lr.fit(X_train_scaled, y_train)

# --------------------------------------
# PREDICTIONS
# --------------------------------------
y_train_pred = lr.predict(X_train_scaled)
y_test_pred = lr.predict(X_test_scaled)

# --------------------------------------
# METRICS (PERCENTAGE)
# --------------------------------------
train_accuracy = accuracy_score(y_train, y_train_pred) * 100
test_accuracy = accuracy_score(y_test, y_test_pred) * 100

precision = precision_score(y_test, y_test_pred) * 100
recall = recall_score(y_test, y_test_pred) * 100

print("\n--- Logistic Regression Performance ---")
print(f"Training Accuracy : {train_accuracy:.2f}%")
print(f"Testing Accuracy  : {test_accuracy:.2f}%")
print(f"Precision         : {precision:.2f}%")
print(f"Recall            : {recall:.2f}%")

# --------------------------------------
# OVERFITTING CHECK
# --------------------------------------
print("\n--- Overfitting Analysis ---")
if train_accuracy - test_accuracy > 5:
    print("⚠️ Possible Overfitting Detected")
else:
    print("✅ No Overfitting (Good Generalization)")

# --------------------------------------
# CONFUSION MATRIX
# --------------------------------------
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Cancer','Cancer'],
            yticklabels=['Non-Cancer','Cancer'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

# --------------------------------------
# BAR CHART OF METRICS
# --------------------------------------
metrics = ['Train Accuracy', 'Test Accuracy', 'Precision', 'Recall']
values = [train_accuracy, test_accuracy, precision, recall]

plt.figure(figsize=(7,4))
plt.bar(metrics, values)
plt.ylim(0,100)
plt.ylabel("Percentage (%)")
plt.title("Logistic Regression Performance Metrics")
plt.show()

# ======================================
# MANUAL SINGLE PATIENT INPUT
# ======================================

print("\n--- Enter Single Patient Details ---")

age = float(input("Age: "))
sex = input("Sex (M/F): ").strip().upper()
stage = float(input("Stage: "))
plasma_C = float(input("Plasma C: "))
creatinine = float(input("Creatinine: "))
LYVE1 = float(input("LYVE1: "))
REG1B = float(input("REG1B: "))
TFF1 = float(input("TFF1: "))
REG1A = float(input("REG1A: "))
CA19_9 = float(input("CA19-9: "))
CEA = float(input("CEA: "))
bilirubin = float(input("Bilirubin: "))
glucose = float(input("Glucose: "))
urine_volume = float(input("Urine Volume: "))
urine_ph = float(input("Urine pH: "))

sex = 0 if sex == 'M' else 1

single_patient = pd.DataFrame([[age, sex, stage, plasma_C, creatinine, LYVE1,
                                 REG1B, TFF1, REG1A, CA19_9, CEA, bilirubin,
                                 glucose, urine_volume, urine_ph]],
                               columns=X.columns)

single_patient_scaled = scaler.transform(single_patient)

prediction = lr.predict(single_patient_scaled)[0]

print("\n--- Prediction Result ---")
if prediction == 1:
    print("🔴 PANCREATIC CANCER")
else:
    print("🟢 NON-CANCER")